# GrainSight AI: Phase 2 Conditional Diffusion Augmentation (v2.0)
### Project: Multi-Food-Grain Quality Analysis Using Diffusion Models and Deep Learning
**Objective**: Train a class-conditional 2D DDPM U-Net with unconditional dropout and Classifier-Free Guidance (CFG) on rare rice-defect classes (`2_SD`: Sprouted Damage, `1_F&S`: Fusarium & Scab) using only the 1,672 session-grouped training images.

---
### Key Architectural Corrections in v2.0:
1. **Unconditional Class Dropout (15%)**: Added `UNCONDITIONAL_NULL = 2` label embedding. Model learns both conditional $\epsilon_	heta(x_t, t, c)$ and unconditional $\epsilon_	heta(x_t, t, \emptyset)$ scores.
2. **True Classifier-Free Guidance (CFG)**: Reverse sampling applies $\hat{\epsilon} = \epsilon_\emptyset + w(\epsilon_c - \epsilon_\emptyset)$ with $w \in [2.0, 3.5]$ to eliminate random noise output.
3. **Comprehensive Checkpointing**: Saves model weights, optimizer state, global step, loss history, config, and RNG seeds. Supports seamless training resumption.
4. **Periodic Visual Monitoring**: Automatically logs a 4-image visual inspection grid at defined checkpoint intervals so grain emergence can be visually confirmed before stopping.
5. **Zero-Leakage Assurance**: Strictly loads 1,672 training images from `splits/leak_free_split.csv`. Validation (2,472) and Test (4,953) remain 100% frozen.


## 1. Hardware & GPU Environment Verification
Verifies that a CUDA-capable GPU (e.g. NVIDIA T4 on Google Colab) is active. Fails immediately if only CPU is detected.


In [ ]:
import sys
import torch

print("=" * 65)
print("GRAINSIGHT AI (v2.0): GPU HARDWARE AUDIT")
print("=" * 65)

if not torch.cuda.is_available():
    raise RuntimeError(
        "[CRITICAL HARDWARE ERROR] No CUDA GPU detected!\n"
        "Diffusion training on CPU is computationally infeasible.\n"
        "In Google Colab, select: Runtime -> Change runtime type -> T4 GPU, then rerun."
    )

gpu_idx = torch.cuda.current_device()
gpu_name = torch.cuda.get_device_name(gpu_idx)
gpu_props = torch.cuda.get_device_properties(gpu_idx)
vram_gb = gpu_props.total_memory / (1024 ** 3)
cuda_version = torch.version.cuda

print(f"[OK] Active GPU Device   : {gpu_name}")
print(f"[OK] Dedicated VRAM     : {vram_gb:.2f} GB")
print(f"[OK] PyTorch Version    : {torch.__version__}")
print(f"[OK] CUDA Driver Version: {cuda_version}")
print("=" * 65)
print("Hardware audit PASSED. Ready for diffusion operations.")


## 2. Environment Setup & Dependencies (Gated)
Installs required packages (`diffusers`, `accelerate`, `torchvision`, `pandas`, `pillow`, `matplotlib`).
Set `INSTALL_DEPENDENCIES = True` when running for the first time in a fresh Colab runtime.


In [ ]:
# SAFETY TOGGLE: Set to True to install missing packages
INSTALL_DEPENDENCIES = False

import importlib
import subprocess

packages = ["diffusers", "accelerate", "torchvision", "PIL", "pandas", "matplotlib"]
missing = []

for pkg in packages:
    try:
        importlib.import_module(pkg)
    except ImportError:
        missing.append(pkg)

if missing:
    print(f"Detected missing packages: {missing}")
    if INSTALL_DEPENDENCIES:
        print("INSTALL_DEPENDENCIES is True. Installing packages...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "diffusers", "accelerate", "torchvision", "pandas", "pillow", "matplotlib"])
        print("[OK] Dependencies installed successfully.")
    else:
        print("[SAFEGUARD ACTIVE] INSTALL_DEPENDENCIES is False. Set to True to install.")
else:
    print("[OK] All required packages are already available.")


## 3. Data Ingestion & Strict Zero-Leakage Audit
Loads `splits/leak_free_split.csv` and verifies that all 1,672 training images for classes `2_SD` and `1_F&S` exist.
Fails early if any single image is missing. Held-out validation and test sets remain completely excluded.


In [ ]:
import os
import pandas as pd
from PIL import Image

MANIFEST_PATH = "splits/leak_free_split.csv"
DATA_ROOT = "."

if not os.path.exists(MANIFEST_PATH):
    if os.path.exists(os.path.join("..", MANIFEST_PATH)):
        MANIFEST_PATH = os.path.join("..", MANIFEST_PATH)
        DATA_ROOT = ".."
    else:
        raise FileNotFoundError(
            f"Manifest not found at {MANIFEST_PATH}. Ensure rice_colab_train_bundle.zip is extracted at project root."
        )

# Read full split manifest
full_df = pd.read_csv(MANIFEST_PATH)
print(f"Total manifest records: {len(full_df)}")

# 1. Zero-Leakage Filtering: TRAIN split ONLY
TARGET_CLASSES = ["2_SD", "1_F&S"]
train_df = full_df[
    (full_df["split"] == "train") & 
    (full_df["class_name"].isin(TARGET_CLASSES))
].copy().reset_index(drop=True)

# 2. Assert Zero Leakage
val_overlap = full_df[(full_df["split"] == "val") & (full_df["class_name"].isin(TARGET_CLASSES))]
test_overlap = full_df[(full_df["split"] == "test") & (full_df["class_name"].isin(TARGET_CLASSES))]

print("=" * 65)
print("ZERO-LEAKAGE MANIFEST VERIFICATION")
print("=" * 65)
print(f"Training samples selected : {len(train_df)} kernels")
for c in TARGET_CLASSES:
    count = (train_df["class_name"] == c).sum()
    print(f"  - Class {c:6s}: {count:4d} training samples")

print(f"Frozen Val excluded      : {len(val_overlap):4d} kernels (STRICTLY FROZEN)")
print(f"Frozen Test excluded     : {len(test_overlap):4d} kernels (STRICTLY FROZEN)")

assert len(train_df) == 1672, f"Expected 1672 training samples, got {len(train_df)}"
assert (train_df["split"] != "train").sum() == 0, "LEAKAGE ERROR: Non-train split found!"
print("[OK] Zero-leakage assertion verified.")

# 3. Path Validation - Fail early if any file is missing
valid_paths = []
dimensions = []
missing_paths = []

for idx, row in train_df.iterrows():
    clean_rel = row["relative_path"].replace("\\", "/")
    img_path = os.path.join(DATA_ROOT, clean_rel)
    if os.path.exists(img_path):
        valid_paths.append(img_path)
        with Image.open(img_path) as img:
            dimensions.append(img.size)
    else:
        missing_paths.append((row["relative_path"], img_path))

if missing_paths:
    raise FileNotFoundError(
        f"[CRITICAL DATA ERROR] {len(missing_paths)} required images missing!\n"
        f"First missing: {missing_paths[0]}\n"
        "Ensure rice_colab_train_bundle.zip was created via scripts/package_colab_bundle.py and extracted."
    )

train_df["full_path"] = valid_paths
assert len(train_df) == len(valid_paths), "Dataframe row count mismatch!"
print(f"[OK] Successfully verified all {len(valid_paths)} training image paths on disk.")


## 4. Architecture Definition & Unconditional Class Dropout
### The CFG Fix:
To enable Classifier-Free Guidance (CFG), the U-Net must learn both conditional and unconditional distributions.
We configure `num_class_embeds = 3`:
- `0`: Class `1_F&S` (Fusarium & Scab)
- `1`: Class `2_SD` (Sprouted Damage)
- `2`: `UNCONDITIONAL_NULL` (Null label $\emptyset$)

During training, with probability $p = 0.15$ (15%), the true class is randomly replaced with the null label `2`.


In [ ]:
import random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torchvision.transforms as T
import torch.nn.functional as F
from diffusers import UNet2DModel, DDPMScheduler, DDIMScheduler

IMAGE_SIZE = 128
CLASS_TO_ID = {"1_F&S": 0, "2_SD": 1}
ID_TO_CLASS = {0: "1_F&S", 1: "2_SD"}
UNCOND_NULL_ID = 2  # Reserved index for unconditional null embedding
NUM_CLASS_EMBEDS = 3  # 0: 1_F&S, 1: 2_SD, 2: Null
UNCOND_DROPOUT_PROB = 0.15  # 15% class dropout for CFG

class SquarePad:
    def __call__(self, img):
        w, h = img.size
        max_dim = max(w, h)
        pad_left = (max_dim - w) // 2
        pad_top = (max_dim - h) // 2
        pad_right = max_dim - w - pad_left
        pad_bottom = max_dim - h - pad_top
        return T.functional.pad(img, (pad_left, pad_top, pad_right, pad_bottom), fill=0)

diffusion_transforms = T.Compose([
    SquarePad(),
    T.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    T.RandomHorizontalFlip(p=0.5),
    T.RandomVerticalFlip(p=0.5),
    T.ToTensor(),
    T.Normalize(mean=[0.5, 0.5, 0.5], std=[0.5, 0.5, 0.5])  # Map to [-1, 1]
])

class RiceDiffusionDatasetV2(Dataset):
    def __init__(self, df, transform=None, uncond_prob=0.15):
        self.df = df
        self.transform = transform
        self.uncond_prob = uncond_prob
        
    def __len__(self):
        return len(self.df)
        
    def __getitem__(self, idx):
        row = self.df.iloc[idx]
        img_path = row["full_path"]
        img = Image.open(img_path).convert("RGB")
        true_label = CLASS_TO_ID[row["class_name"]]
        
        # Unconditional class dropout during training
        if self.uncond_prob > 0 and random.random() < self.uncond_prob:
            effective_label = UNCOND_NULL_ID
        else:
            effective_label = true_label
            
        if self.transform:
            img = self.transform(img)
            
        return img, effective_label, true_label

train_dataset_v2 = RiceDiffusionDatasetV2(train_df, transform=diffusion_transforms, uncond_prob=UNCOND_DROPOUT_PROB)
print(f"[OK] RiceDiffusionDatasetV2 initialized with {len(train_dataset_v2)} samples.")
print(f"Class mapping: {CLASS_TO_ID}, Null ID: {UNCOND_NULL_ID}, Uncond Dropout: {UNCOND_DROPOUT_PROB*100:.0f}%")


## 5. Small Pilot Verification Mode (Gated)
Runs 1 epoch on a single batch of 4 images before starting any long GPU run.
Verifies forward pass, backward pass, loss computation, peak VRAM, and model construction.
Set `RUN_PILOT = True` to execute.


In [ ]:
# SAFETY TOGGLE: Set to True to execute pilot verification
RUN_PILOT = False

if not RUN_PILOT:
    print("[SAFEGUARD ACTIVE] RUN_PILOT is False.")
    print("Set `RUN_PILOT = True` to run the 1-batch verification step.")
else:
    print("=" * 65)
    print("GRAINSIGHT AI: EXECUTING SMALL PILOT VERIFICATION")
    print("=" * 65)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # Initialize U-Net with num_class_embeds = 3
    pilot_model = UNet2DModel(
        sample_size=IMAGE_SIZE,
        in_channels=3,
        out_channels=3,
        layers_per_block=2,
        block_out_channels=(64, 128, 256, 256),
        down_block_types=(
            "DownBlock2D",
            "DownBlock2D",
            "AttnDownBlock2D",
            "DownBlock2D",
        ),
        up_block_types=(
            "UpBlock2D",
            "AttnUpBlock2D",
            "UpBlock2D",
            "UpBlock2D",
        ),
        num_class_embeds=NUM_CLASS_EMBEDS
    ).to(device)
    
    pilot_scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="linear", prediction_type="epsilon")
    pilot_opt = torch.optim.AdamW(pilot_model.parameters(), lr=1e-4)
    
    pilot_loader = DataLoader(train_dataset_v2, batch_size=4, shuffle=True)
    batch_img, batch_eff_lbl, _ = next(iter(pilot_loader))
    batch_img = batch_img.to(device)
    batch_eff_lbl = batch_eff_lbl.to(device)
    
    noise = torch.randn_like(batch_img)
    timesteps = torch.randint(0, 1000, (batch_img.shape[0],), device=device).long()
    noisy_img = pilot_scheduler.add_noise(batch_img, noise, timesteps)
    
    pilot_opt.zero_grad()
    noise_pred = pilot_model(noisy_img, timesteps, class_labels=batch_eff_lbl).sample
    loss = F.mse_loss(noise_pred, noise)
    loss.backward()
    pilot_opt.step()
    
    peak_vram_mb = torch.cuda.max_memory_allocated(device) / (1024 ** 2) if torch.cuda.is_available() else 0.0
    param_count = sum(p.numel() for p in pilot_model.parameters()) / 1e6
    
    print(f"[OK] Pilot Step Loss        : {loss.item():.4f}")
    print(f"[OK] Peak VRAM Consumption  : {peak_vram_mb:.2f} MB")
    print(f"[OK] Model Parameter Count  : {param_count:.2f} Million")
    print("=" * 65)
    print("Pilot verification complete. Model architecture is functional.")


## 6. Checkpointed Training Loop with Visual Sample Grid Logging (Gated)
### Experimental Training Configuration:
- `NUM_EPOCHS`: 80 to 120 (recommended candidate, ~8,300 to 12,500 gradient steps)
- `BATCH_SIZE`: 16
- `LEARNING_RATE`: $1 	imes 10^{-4}$ with AdamW
- `SAMPLE_EVERY_EPOCHS`: Generates and logs a 4-sample visual inspection grid every 20 epochs to track visual grain emergence.
- `RESUME_FROM_CHECKPOINT`: If True, automatically resumes from `checkpoints/grainsight_ddpm_latest.pt`.


In [ ]:
# SAFETY TOGGLE: Set to True to start training on GPU
RUN_FULL_TRAINING = False
RESUME_FROM_CHECKPOINT = False

NUM_EPOCHS = 100
BATCH_SIZE = 16
LEARNING_RATE = 1e-4
SAMPLE_EVERY_EPOCHS = 20
CHECKPOINT_DIR = "checkpoints_v2"
SAMPLE_DIR = "visual_progress_v2"

if not RUN_FULL_TRAINING:
    print("[SAFEGUARD ACTIVE] RUN_FULL_TRAINING is False.")
    print("To execute training on Colab T4 GPU, set `RUN_FULL_TRAINING = True`.")
else:
    import matplotlib.pyplot as plt
    
    os.makedirs(CHECKPOINT_DIR, exist_ok=True)
    os.makedirs(SAMPLE_DIR, exist_ok=True)
    
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    
    # 1. Initialize U-Net model
    model = UNet2DModel(
        sample_size=IMAGE_SIZE,
        in_channels=3,
        out_channels=3,
        layers_per_block=2,
        block_out_channels=(64, 128, 256, 256),
        down_block_types=(
            "DownBlock2D",
            "DownBlock2D",
            "AttnDownBlock2D",
            "DownBlock2D",
        ),
        up_block_types=(
            "UpBlock2D",
            "AttnUpBlock2D",
            "UpBlock2D",
            "UpBlock2D",
        ),
        num_class_embeds=NUM_CLASS_EMBEDS
    ).to(device)
    
    noise_scheduler = DDPMScheduler(num_train_timesteps=1000, beta_schedule="linear", prediction_type="epsilon")
    optimizer = torch.optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=1e-5)
    
    start_epoch = 1
    global_step = 0
    loss_history = []
    
    # 2. Checkpoint Resume Logic
    latest_ckpt_path = os.path.join(CHECKPOINT_DIR, "grainsight_ddpm_latest.pt")
    if RESUME_FROM_CHECKPOINT and os.path.exists(latest_ckpt_path):
        print(f"Resuming from checkpoint: {latest_ckpt_path}")
        ckpt = torch.load(latest_ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model_state_dict"])
        optimizer.load_state_dict(ckpt["optimizer_state_dict"])
        start_epoch = ckpt["epoch"] + 1
        global_step = ckpt["global_step"]
        loss_history = ckpt.get("loss_history", [])
        print(f"Resumed at Epoch {start_epoch}, Global Step {global_step}")
        
    loader = DataLoader(train_dataset_v2, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
    steps_per_epoch = len(loader)
    total_steps = NUM_EPOCHS * steps_per_epoch
    
    print("=" * 65)
    print(f"STARTING GRAINSIGHT AI DIFFUSION TRAINING")
    print(f"Target Epochs: {NUM_EPOCHS} | Steps/Epoch: {steps_per_epoch} | Total Steps: {total_steps}")
    print("=" * 65)
    
    for epoch in range(start_epoch, NUM_EPOCHS + 1):
        model.train()
        epoch_loss = 0.0
        
        for step, (images, eff_labels, _) in enumerate(loader):
            images = images.to(device)
            eff_labels = eff_labels.to(device)
            
            noise = torch.randn_like(images)
            timesteps = torch.randint(0, noise_scheduler.config.num_train_timesteps, (images.shape[0],), device=device).long()
            noisy_images = noise_scheduler.add_noise(images, noise, timesteps)
            
            optimizer.zero_grad()
            noise_pred = model(noisy_images, timesteps, class_labels=eff_labels).sample
            loss = F.mse_loss(noise_pred, noise)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            
            epoch_loss += loss.item()
            global_step += 1
            
        avg_loss = epoch_loss / steps_per_epoch
        loss_history.append({"epoch": epoch, "loss": round(avg_loss, 5), "step": global_step})
        
        if epoch % 5 == 0 or epoch == NUM_EPOCHS:
            print(f"Epoch [{epoch:3d}/{NUM_EPOCHS}] Step [{global_step:5d}] - MSE Loss: {avg_loss:.5f}")
            
        # 3. Save Checkpoint with Complete Metadata
        if epoch % 20 == 0 or epoch == NUM_EPOCHS:
            ckpt_data = {
                "epoch": epoch,
                "global_step": global_step,
                "model_state_dict": model.state_dict(),
                "optimizer_state_dict": optimizer.state_dict(),
                "loss_history": loss_history,
                "config": {
                    "image_size": IMAGE_SIZE,
                    "num_class_embeds": NUM_CLASS_EMBEDS,
                    "uncond_dropout": UNCOND_DROPOUT_PROB,
                    "scheduler": "DDPMScheduler(1000, linear)"
                }
            }
            torch.save(ckpt_data, latest_ckpt_path)
            torch.save(ckpt_data, os.path.join(CHECKPOINT_DIR, f"grainsight_ddpm_epoch_{epoch:03d}.pt"))
            print(f"  --> Checkpoint saved for epoch {epoch}")
            
        # 4. Generate Visual Inspection Grid (4 samples: 2 of each class)
        if epoch % SAMPLE_EVERY_EPOCHS == 0 or epoch == NUM_EPOCHS:
            model.eval()
            ddim = DDIMScheduler.from_config(noise_scheduler.config)
            ddim.set_timesteps(30)
            
            sample_labels = torch.tensor([0, 0, 1, 1], device=device)
            sample_null = torch.tensor([UNCOND_NULL_ID] * 4, device=device)
            x_t = torch.randn((4, 3, IMAGE_SIZE, IMAGE_SIZE), device=device)
            
            guidance_scale = 2.5
            with torch.no_grad():
                for t in ddim.timesteps:
                    # Model evaluation with class and null
                    cond_pred = model(x_t, t, class_labels=sample_labels).sample
                    uncond_pred = model(x_t, t, class_labels=sample_null).sample
                    # Classifier-Free Guidance formula
                    guided_pred = uncond_pred + guidance_scale * (cond_pred - uncond_pred)
                    x_t = ddim.step(guided_pred, t, x_t).prev_sample
                    
            x_vis = (x_t / 2 + 0.5).clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()
            
            fig, axes = plt.subplots(1, 4, figsize=(12, 3))
            titles = ["1_F&S (Sample 1)", "1_F&S (Sample 2)", "2_SD (Sample 1)", "2_SD (Sample 2)"]
            for i, ax in enumerate(axes):
                ax.imshow(x_vis[i])
                ax.set_title(titles[i], fontsize=10)
                ax.axis("off")
            sample_plot_path = os.path.join(SAMPLE_DIR, f"inspection_grid_epoch_{epoch:03d}.png")
            plt.tight_layout()
            plt.savefig(sample_plot_path, dpi=120)
            plt.close()
            print(f"  --> Visual progress grid saved: {sample_plot_path}")
            
    print("=" * 65)
    print("TRAINING RUN COMPLETED. Inspect generated visual progress grids!")


## 7. Controlled Reverse Diffusion Sampling with True CFG (Gated)
Uses DDIM reverse sampling with Classifier-Free Guidance:
$$\hat{\epsilon} = \epsilon_\emptyset + w \cdot (\epsilon_c - \epsilon_\emptyset)$$
- Recommended guidance scale $w$: `2.5` to `3.5`.
- Set `RUN_SYNTHESIS = True` to generate synthetic candidates after visual inspection passes.


In [ ]:
# SAFETY TOGGLE: Set to True after training to sample candidate defect grains
RUN_SYNTHESIS = False
CANDIDATES_PER_CLASS = 25  # Start with 25 per class to visually inspect before scale
CFG_SCALE = 3.0
INFERENCE_STEPS = 50

if not RUN_SYNTHESIS:
    print("[SAFEGUARD ACTIVE] RUN_SYNTHESIS is False.")
    print("Set `RUN_SYNTHESIS = True` to run CFG sampling.")
else:
    import numpy as np

    print("=" * 65)
    print(f"SAMPLING WITH CLASSIFIER-FREE GUIDANCE (w = {CFG_SCALE})")
    print("=" * 65)

    os.makedirs("synthetic_v2/raw", exist_ok=True)
    ddim_sampler = DDIMScheduler.from_config(noise_scheduler.config)
    ddim_sampler.set_timesteps(INFERENCE_STEPS)

    model.eval()
    generated_records = []

    for class_name, class_id in CLASS_TO_ID.items():
        print(f"Synthesizing class: {class_name} (ID: {class_id}) ...")
        labels = torch.tensor([class_id] * CANDIDATES_PER_CLASS, device=device)
        null_labels = torch.tensor([UNCOND_NULL_ID] * CANDIDATES_PER_CLASS, device=device)
        
        # Fixed generator seed per batch for reproducibility
        generator = torch.Generator(device=device).manual_seed(42 + class_id * 1000)
        x_t = torch.randn((CANDIDATES_PER_CLASS, 3, IMAGE_SIZE, IMAGE_SIZE), generator=generator, device=device)
        
        with torch.no_grad():
            for t in ddim_sampler.timesteps:
                cond_pred = model(x_t, t, class_labels=labels).sample
                uncond_pred = model(x_t, t, class_labels=null_labels).sample
                guided_pred = uncond_pred + CFG_SCALE * (cond_pred - uncond_pred)
                x_t = ddim_sampler.step(guided_pred, t, x_t).prev_sample
                
        # Denormalize [-1, 1] -> [0, 255]
        samples = (x_t / 2 + 0.5).clamp(0, 1).cpu().permute(0, 2, 3, 1).numpy()
        
        for i in range(CANDIDATES_PER_CLASS):
            img_arr = (samples[i] * 255).astype(np.uint8)
            img_out = Image.fromarray(img_arr)
            filename = f"synth_{class_name}_cfg_{CFG_SCALE}_seed_{42 + i:04d}.png"
            save_path = os.path.join("synthetic_v2/raw", filename)
            img_out.save(save_path)
            
            generated_records.append({
                "image_id": filename,
                "target_class": class_name,
                "guidance_scale": CFG_SCALE,
                "sampling_steps": INFERENCE_STEPS,
                "seed": 42 + i,
                "raw_path": save_path
            })
            
    print(f"[OK] Successfully synthesized {len(generated_records)} raw candidates with CFG.")


## 8. Visual Quality Gate & Dual Morphological Filter (Gated)
Does NOT automatically approve images based on classifier confidence alone.
Requires:
1. Corner background darkness check ($RGB < 25$).
2. Minimum grain segmentation area (>800 px) and natural aspect ratio ($1.8 \le L/W \le 4.8$).
3. Exports `synthetic_provenance_manifest_v2.csv` for human inspection.


In [ ]:
# SAFETY TOGGLE: Set to True to filter generated candidates
RUN_FILTERING = False

if not RUN_FILTERING:
    print("[SAFEGUARD ACTIVE] RUN_FILTERING is False.")
    print("Set `RUN_FILTERING = True` to execute quality filtering.")
else:
    import cv2
    import numpy as np

    os.makedirs("synthetic_v2/approved", exist_ok=True)
    manifest_rows = []

    for rec in generated_records:
        raw_path = rec["raw_path"]
        img_pil = Image.open(raw_path).convert("RGB")
        img_np = np.array(img_pil)
        
        # Check 1: Corner background darkness
        corners = [
            img_np[:10, :10], img_np[:10, -10:],
            img_np[-10:, :10], img_np[-10:, -10:]
        ]
        mean_corner = np.mean([np.mean(c) for c in corners])
        bg_passed = bool(mean_corner < 25.0)
        
        # Check 2: Grain contour & aspect ratio
        gray = cv2.cvtColor(img_np, cv2.COLOR_RGB2GRAY)
        _, thresh = cv2.threshold(gray, 25, 255, cv2.THRESH_BINARY)
        contours, _ = cv2.findContours(thresh, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        
        shape_passed = False
        if contours:
            largest = max(contours, key=cv2.contourArea)
            area = cv2.contourArea(largest)
            if area > 800:
                x, y, w, h = cv2.boundingRect(largest)
                aspect = max(w, h) / (min(w, h) + 1e-5)
                shape_passed = bool(1.8 <= aspect <= 4.8)
                
        is_approved = bg_passed and shape_passed
        status = "APPROVED" if is_approved else "REJECTED"
        
        approved_path = ""
        if is_approved:
            approved_path = os.path.join("synthetic_v2/approved", rec["image_id"])
            img_pil.save(approved_path)
            
        manifest_rows.append({
            "image_id": rec["image_id"],
            "target_class": rec["target_class"],
            "guidance_scale": rec["guidance_scale"],
            "sampling_steps": rec["sampling_steps"],
            "seed": rec["seed"],
            "bg_check_passed": bg_passed,
            "shape_check_passed": shape_passed,
            "filter_status": status,
            "output_path": approved_path
        })

    provenance_df = pd.DataFrame(manifest_rows)
    provenance_path = "synthetic_v2/synthetic_provenance_manifest_v2.csv"
    provenance_df.to_csv(provenance_path, index=False)

    print("=" * 65)
    print("GRAINSIGHT AI: QUALITY FILTERING AUDIT")
    print("=" * 65)
    print(provenance_df["filter_status"].value_counts())
    print(f"Provenance manifest saved to: {provenance_path}")


## 9. Packaging Approved Synthetic Outputs (Gated)
Compresses approved synthetic images and provenance manifest into `grainsight_diffusion_augmented_v2.zip` for download back to your local repository.


In [ ]:
# SAFETY TOGGLE: Set to True to package approved outputs
PACKAGE_OUTPUTS = False

if not PACKAGE_OUTPUTS:
    print("[SAFEGUARD ACTIVE] PACKAGE_OUTPUTS is False.")
    print("Set `PACKAGE_OUTPUTS = True` to package approved outputs into a zip file.")
else:
    import shutil
    out_zip = "grainsight_diffusion_augmented_v2.zip"
    if os.path.exists("synthetic_v2/approved") and len(os.listdir("synthetic_v2/approved")) > 0:
        shutil.make_archive("grainsight_diffusion_augmented_v2", 'zip', "synthetic_v2")
        print(f"[OK] Artifacts compressed into {out_zip}")
    else:
        print("[INFO] No approved synthetic images available to package yet.")
